# Session 0 — Prepare the human lymph-node tutorial

Run this notebook once before the workshop. It downloads the official 10x `V1_Human_Lymph_Node` count matrix and spatial/H&E bundle, copies the tracked germinal-center annotation, and installs the **validated precomputed DGAT prediction matrix** used by participants.

The released model weights are not required for the default hands-on path. They are downloaded only if a participant chooses the optional full-inference section in Session 2.

## 1. Mount Drive and fetch the tutorial repository(estimated_runtime 1min)


In [1]:
from pathlib import Path
import importlib.util, os, subprocess, sys
if importlib.util.find_spec("google.colab") is None: raise RuntimeError("Open Session 0 in Google Colab.")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
repo_dir = Path("/content/ECCB-2026-Tutorial"); tutorial_root = repo_dir / "hands-on_tutorial"
if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
os.chdir(tutorial_root); print("Tutorial repository:", tutorial_root)


Mounted at /content/drive
Tutorial repository: /content/ECCB-2026-Tutorial/hands-on_tutorial


## 2. Download the lymph-node inputs and validated predictions(estimated_runtime 7sec)

For organizer testing, the downloader can use the bundle produced in `MyDrive/ECCB2026/organizer/lymph_node_prediction_build`. For the published tutorial, the same CSV and metadata sidecar should be tracked under `hands-on_tutorial/data/raw`.

In [2]:
drive_root = Path("/content/drive/MyDrive/ECCB2026")
asset_root = drive_root / "assets" / "DGAT_assets"; asset_root.mkdir(parents=True, exist_ok=True)
organizer_output = drive_root / "organizer" / "lymph_node_prediction_build"
env = os.environ.copy()
env["DGAT_ASSET_DIR"] = str(asset_root)
env["DGAT_PRECOMPUTED_DIR"] = str(organizer_output)
cmd = ["bash", "scripts/download_dgat_assets.sh", "--dataset", "V1_Human_Lymph_Node"]
subprocess.run(cmd, cwd=tutorial_root, env=env, check=True)
subprocess.run(cmd + ["--check-only"], cwd=tutorial_root, env=env, check=True)

CompletedProcess(args=['bash', 'scripts/download_dgat_assets.sh', '--dataset', 'V1_Human_Lymph_Node', '--check-only'], returncode=0)

## 3. Verify files and record checksums(estimated_runtime 1sec)

The prediction CSV and its metadata are part of the input contract. Session 2 will independently re-check their sample name, barcode order, protein order, dimensions, and checksum.

In [3]:
import hashlib, json
files = [
    asset_root / "data/V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5",
    asset_root / "data/V1_Human_Lymph_Node_spatial.tar.gz",
    asset_root / "data/V1_Human_Lymph_Node_manual_GC_annot.csv",
    asset_root / "data/spatial/tissue_positions_list.csv",
    asset_root / "data/spatial/tissue_hires_image.png",
    asset_root / "data/V1_Human_Lymph_Node_DGAT_predicted_proteins.csv",
    asset_root / "data/V1_Human_Lymph_Node_DGAT_predicted_proteins.metadata.json",
]
manifest = {"dataset": "V1_Human_Lymph_Node", "source": "10x Visium 1.1.0 plus organizer-validated DGAT predictions", "python": sys.version.split()[0], "files": {}}
for path in files:
    if not path.is_file() or path.stat().st_size == 0: raise FileNotFoundError(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""): digest.update(chunk)
    key = str(path.relative_to(asset_root / "data")) if path.is_relative_to(asset_root / "data") else str(path.relative_to(asset_root))
    manifest["files"][key] = {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}
manifest_path = drive_root / "asset_manifest.json"; manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))

{
  "dataset": "V1_Human_Lymph_Node",
  "source": "10x Visium 1.1.0 plus organizer-validated DGAT predictions",
  "python": "3.12.13",
  "files": {
    "V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5": {
      "bytes": 30736856,
      "sha256": "86fd533eb907450e7125b9820183a0ca73776eeafcc5eddae5695b6aabfd9139"
    },
    "V1_Human_Lymph_Node_spatial.tar.gz": {
      "bytes": 8246238,
      "sha256": "812808883366ff9623dc8354847a7211b0d922b2bfc4c9359d6e12e993ea6a73"
    },
    "V1_Human_Lymph_Node_manual_GC_annot.csv": {
      "bytes": 8327,
      "sha256": "7e3f0d1d32fd95d51fe4ff66752e1850bc2717204359d62860c8bd966c80c8ba"
    },
    "spatial/tissue_positions_list.csv": {
      "bytes": 184912,
      "sha256": "32e0c92dc99f25f66e7512f7078b28ba2988198ea4d1d2a1c4f58ca429cde334"
    },
    "spatial/tissue_hires_image.png": {
      "bytes": 4683026,
      "sha256": "aa5aea6f22519dca0b7dfec1d3ea34280f1ca4c9d8a5ecb04295347f2cbbf065"
    },
    "V1_Human_Lymph_Node_DGAT_predicted_proteins.c

## 4. Cache the participant environment and create restart-safe folders(estimated_runtime 45sec)

The default workflow is CPU-friendly because it loads predictions rather than running the neural network. Session 2 installs GPU-related packages only when its optional reproducibility flag is enabled.

In [4]:
wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"; wheelhouse.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--only-binary=:all:", "--dest", str(wheelhouse), "-r", str(tutorial_root / "requirements-colab.txt")], check=True)
for relative in ("state/data/processed", "state/results/figures", "state/checkpoints"): (drive_root / relative).mkdir(parents=True, exist_ok=True)
print("Drive preparation complete:", drive_root)


Drive preparation complete: /content/drive/MyDrive/ECCB2026
